# 可复现关键流程
运行同一套根入口。已有设计池→代理评分→筛选→候选导出；合成对接只做软件检查，透皮为独立参考载体前向模型。不重新训练。先按README安装requirements。输出必须新目录。

In [ ]:
from pathlib import Path
import subprocess, sys, tempfile, json, csv
project = Path.cwd() if (Path.cwd()/"predict.py").exists() else Path.cwd().parent
assert (project/"predict.py").exists()
work = Path(tempfile.mkdtemp(prefix="submission_demo_"))
def run(script, *args):
    subprocess.run([sys.executable, "-B", str(project/script), *map(str,args)], cwd=project, check=True)


## 完整三模块演示
默认执行AI已有池的评分筛选、明确标注的合成筛选演示及透皮前向模拟。人工mock不会进入正式候选CSV。

In [ ]:
run("demo.py", "--output-dir", work/"demo")
with (work/"demo/results.csv").open(encoding="utf-8-sig",newline="") as f:
    candidates=list(csv.DictReader(f))
assert len(candidates)==84
assert all(r["evidence_type"]=="model_prediction" for r in candidates)
[{k:r[k] for k in ("candidate_id","prediction_value","prediction_unit","model_version")} for r in candidates[:4]]


## 再现校验
核对标准字段、模型结果和科学/合成分离。完整权重复核可加--models。

In [ ]:
run("verify_submission.py")
summary=json.loads((work/"demo/transport/summary.json").read_text())
summary["checks"]


## 真实研究输入
真实docking需自行提供合规受体/配体和Vina；训练用train.py明确选择历史MLP或新版benchmark。赛道已确认为赛道四：递送载体设计；原创数据公开授权与第三方许可见提交核对表。